In [1]:
import pandas as pd
import numpy as np
from src.model_pipeline import ModelPipeline

In [2]:
df_final = pd.read_parquet("data/ready_to_train/DEBIT_df_final_81_feats_20250629.parquet")

In [3]:
categorical_cols = [
    # 'AccountStatus',
    # 'Currency',
    # 'CustomerSex',
    'HIghRiskCustomer',
    'POSMode',
    # 'TransactionType',
    # 'CardProductGrouped',
    # 'CountryCodeGroup'
]
numerical_cols = [
    col for col in df_final.columns if col not in (categorical_cols + ['Confirmed','Transaction Datetime'])
]

In [4]:
top_numerical_cols = [
    'Sum_Amt_L30D',
    'TxnCount_L30D',
    'CountTrxTrf', # TSCF
    'TotalTrxAmount10Mi', # TSCF
    'TotalTrxAmountL5min', # TSCF
    'TotalTrxAmount15Mi', # TSCF
    'CntUnique_CardNo_by_MCC_L30D',
    'TotalTrxAmountL1D', # TSCF
    'Transaction Amount', # Channel
    'CustomerAge', # TSCF
    'RatioTrxAmountL1DL15min', # TSCF
    'RatioTrxAmountL1DL10min', # TSCF
    'RatioTrxAmountL1DL5min', # TSCF
    'Max_Amt_L30D',
    'Avg_Amt_L30D',
    'RatioCntUnique_CardNo_by_MCCL30DL15min',
    'RatioTxnCountL30DL15min',
    'AvgTrnxHourL30d',
    'time_diff',
    'DurationSinceFirstTrnxToCurrentMCC',
    'AvgDurationSinceFirstTrnxToCurrentMCCL30D',
    'AvgTrnxHourL15min',
    'CntUnique_CardNo_by_MCC_L15M',
    # 'Avg_Amt_to_MCC_L30D',
    # 'Max_Amt_to_MCC_L30D'
]

In [17]:
len(top_numerical_cols + categorical_cols)

25

In [6]:
df_final = df_final[['Confirmed','Transaction Datetime'] + top_numerical_cols + categorical_cols]

# Rename Column

In [7]:
renamed_column_map = {
    'AvgDurationSinceFirstTrnxToCurrentMCCL30D': 'AvgTimeFirstTxnToCurrentMCCL30D',
    'Avg_Amt_L30D': 'AvgAmtL30D',
    'Avg_Amt_to_MCC_L30D': 'AvgAmtToMCCL30D',
    'CntUnique_CardNo_by_MCC_L15M': 'CntUniqueCardNoByMCCL15min',
    'CntUnique_CardNo_by_MCC_L30D': 'CntUniqueCardNoByMCCL30D',
    'DurationSinceFirstTrnxToCurrentMCC': 'TimeFirstTxnToCurrentMCC',
    'HIghRiskCustomer': 'HighRiskCustomer',
    'Max_Amt_L30D': 'MaxAmtL30D',
    'Max_Amt_to_MCC_L30D': 'MaxAmtToMCCL30D',
    'RatioCntUnique_CardNo_by_MCCL30DL15min': 'RatioCntUniqueCardNoByMCCL30DL15min',
    'Sum_Amt_L30D': 'SumAmtL30D',
    'Transaction Amount': 'TransactionAmount',
    'TxnCount_L30D': 'TxnCountL30D',
    'time_diff': 'TxnTimeDifference',
}
df_final = df_final.rename(columns=renamed_column_map)

In [8]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 714734 entries, 1047361273 to 1143344726
Data columns (total 27 columns):
 #   Column                               Non-Null Count   Dtype         
---  ------                               --------------   -----         
 0   Confirmed                            714734 non-null  float64       
 1   Transaction Datetime                 714734 non-null  datetime64[ns]
 2   SumAmtL30D                           714734 non-null  float64       
 3   TxnCountL30D                         714734 non-null  float64       
 4   CountTrxTrf                          385721 non-null  float64       
 5   TotalTrxAmount10Mi                   680394 non-null  float64       
 6   TotalTrxAmountL5min                  679661 non-null  float64       
 7   TotalTrxAmount15Mi                   680838 non-null  float64       
 8   CntUniqueCardNoByMCCL30D             188090 non-null  float64       
 9   TotalTrxAmountL1D                    689679 non-null  float64 

# Model Train

In [29]:
df_final.to_parquet("data/ready_to_train/df_debit_final_25_feats_20250702.parquet")

In [9]:
# setup pipeline
pipeline = ModelPipeline(
    model_type="random_forest",
    random_state=42
)

split_date = "2025-05-01"
df_splits = pipeline.split_data_by_date(
    df=df_final,
    date_column="Transaction Datetime",
    split_date=split_date
) 

Initializing ModelPipeline with model type: random_forest
Retrieving model instance for type: random_forest
Splitting data by date (pre-preprocessing)...
Train samples: 647760, Test samples: 66974
Data splitting complete.


In [10]:
# prepare TRAIN data
target_col = "Confirmed"
X, y = pipeline.prepare_data(
    df=df_splits["df_train"],
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=True
)

Preparing data (is_training=True)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Fitting CategoryManager...
  Fitted categories for column 'HighRiskCustomer': ['0', 'N', '__missing__']
  Fitted categories for column 'POSMode': ['0', '1', '2', '5', '7', '9', '__missing__']
CategoryManager fitting complete.
Transforming data using CategoryManager...
CategoryManager transformation complete.
Applying One-Hot Encoders for column: Index(['HighRiskCustomer', 'POSMode'], dtype='object')...
Identified numerical columns for imputation: ['SumAmtL30D', 'TxnCountL30D', 'CountTrxTrf', 'TotalTrxAmount10Mi', 'TotalTrxAmountL5min', 'TotalTrxAmount15Mi', 'CntUniqueCardNoByMCCL30D', 'TotalTrxAmountL1D', 'TransactionAmount', 'CustomerAge', 'RatioTrxAmountL1DL15min', 'RatioTrxAmountL1DL10min', 'RatioTrxAmountL1DL5min', 'MaxAmtL30D', 'AvgAmtL30D', 'RatioCntUniqueCardNoByMCCL30DL15min', 'RatioTxnCountL30DL15min', 'AvgTrnxHourL30d', 'TxnTimeDifference', 

In [11]:
len(X.columns)

31

In [12]:
# split data
X_train, X_test, y_train, y_test = pipeline.split_data(X, y, test_size=0.3)

Splitting data into train and test sets...
Data splitting complete.


## Randomized

In [14]:
from src.imbalance_learn import ImbalancedSampler

# OverSampling
ros_sampler = ImbalancedSampler(
    algorithm='RandomOverSampler',
    sampler_type='upsample',
    target_ratio=0.2, # 7%
    random_state=1234
)

X_train_ros, y_train_ros = ros_sampler.fit_resample(X_train, y_train)

Original dataset shape: Counter({0.0: 453118, 1.0: 314})
Resampled dataset shape using RandomOverSampler: Counter({0.0: 453118, 1.0: 90623})


## SMOTE

In [16]:
# OverSampling
smote_sampler = ImbalancedSampler(
    algorithm='SMOTE',
    sampler_type='upsample',
    target_ratio=0.2, # 7%
    random_state=1234
)

X_train_smote, y_train_smote = smote_sampler.fit_resample(X_train, y_train)

Original dataset shape: Counter({0.0: 453118, 1.0: 314})
Resampled dataset shape using SMOTE: Counter({0.0: 453118, 1.0: 90623})


## Train ROS

In [19]:
# build and train
pipeline.build_pipeline()
pipeline.train(X_train_ros, y_train_ros, show_training_log=False)

Building pipeline...
Pipeline built.
Training model...
Model training complete.


In [20]:
base_rf_ros = pipeline.get_model()

Returning trained model...


In [27]:
import pickle

filepath = "model/debit/base_rf_ros_31_feats_20250702.pkl"
with open(filepath, "wb") as f:
    pickle.dump(base_rf_ros, f)

In [21]:
# Evaluate
test_results = pipeline.evaluate(X_test, y_test)

Evaluating model...
Accuracy: 0.9825
AUC: 0.9674
PR AUC: 0.0956
Precision: 0.0251
Recall: 0.6370

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      0.98      0.99    194193
         1.0       0.03      0.64      0.05       135

    accuracy                           0.98    194328
   macro avg       0.51      0.81      0.52    194328
weighted avg       1.00      0.98      0.99    194328



## Train SMOTE

In [18]:
# build and train
pipeline.build_pipeline()
pipeline.train(X_train_smote, y_train_smote, show_training_log=False)

Building pipeline...
Pipeline built.
Training model...



KeyboardInterrupt



In [23]:
base_rf_smote = pipeline.get_model()

Returning trained model...


In [24]:
# Evaluate
test_results_smote = pipeline.evaluate(X_test, y_test)

Evaluating model...
Accuracy: 0.9868
AUC: 0.9678
PR AUC: 0.1305
Precision: 0.0302
Recall: 0.5778

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      0.99      0.99    194193
         1.0       0.03      0.58      0.06       135

    accuracy                           0.99    194328
   macro avg       0.51      0.78      0.53    194328
weighted avg       1.00      0.99      0.99    194328



# Backtest

In [22]:
df_debit_mix = pd.read_parquet("data/base/df_debit_clean_v2.parquet")
trx_id_list = list(df_debit_mix[df_debit_mix.Confirmed.isin([0,1])]['Transaction Serial No'])
df_oos_new = df_splits["df_test"][df_splits["df_test"].index.isin(trx_id_list)]

In [23]:
X_oos, y_oos = pipeline.prepare_data(
    df=df_splits["df_test"],
    # df=df_oos_new,
    target_column=target_col,
    exclude_columns=["Transaction Datetime"],
    is_apply_one_hot=True,
    is_apply_log=False,
    is_impute_median=False,
    is_training=False
)

Preparing data (is_training=False)...
Starting core numeric and boolean preprocessing...
Core numeric and boolean preprocessing complete.
Transforming data using CategoryManager...
CategoryManager transformation complete.
Applying One-Hot Encoders for column: Index(['HighRiskCustomer', 'POSMode'], dtype='object')...
Identified numerical columns for imputation: ['SumAmtL30D', 'TxnCountL30D', 'CountTrxTrf', 'TotalTrxAmount10Mi', 'TotalTrxAmountL5min', 'TotalTrxAmount15Mi', 'CntUniqueCardNoByMCCL30D', 'TotalTrxAmountL1D', 'TransactionAmount', 'CustomerAge', 'RatioTrxAmountL1DL15min', 'RatioTrxAmountL1DL10min', 'RatioTrxAmountL1DL5min', 'MaxAmtL30D', 'AvgAmtL30D', 'RatioCntUniqueCardNoByMCCL30DL15min', 'RatioTxnCountL30DL15min', 'AvgTrnxHourL30d', 'TxnTimeDifference', 'TimeFirstTxnToCurrentMCC', 'AvgTimeFirstTxnToCurrentMCCL30D', 'AvgTrnxHourL15min', 'CntUniqueCardNoByMCCL15min']
Identified categorical columns (encoded): []
Transforming data using NumericalImputer...
  Imputed column 'SumA

In [24]:
# Evaluate ALL trx (base RF ROS)
oos_results = pipeline.evaluate(X_oos, y_oos)

Evaluating model...
Accuracy: 0.9860
AUC: 0.9810
PR AUC: 0.1192
Precision: 0.0304
Recall: 0.7436

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      0.99      0.99     66935
         1.0       0.03      0.74      0.06        39

    accuracy                           0.99     66974
   macro avg       0.52      0.86      0.53     66974
weighted avg       1.00      0.99      0.99     66974



## SMOTE Backtest

In [25]:
# SMOTE RF
oos_results_smote = pipeline.evaluate(X_oos, y_oos)

Evaluating model...
Accuracy: 0.9898
AUC: 0.9788
PR AUC: 0.1396
Precision: 0.0387
Recall: 0.6923

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      0.99      0.99     66935
         1.0       0.04      0.69      0.07        39

    accuracy                           0.99     66974
   macro avg       0.52      0.84      0.53     66974
weighted avg       1.00      0.99      0.99     66974



# Feature Importance

In [22]:
# Feature importance
pd.set_option("display.max_rows", None)
importance_df = pipeline.get_feature_importance(X_train_ros, y_train_ros)
importance_df['importance'] = np.round(importance_df['importance'], 6)

Getting feature importances...
  Using native feature importances.


In [23]:
importance_df.reset_index(drop=True)

,feature,importance
0,TxnCountL30D,0.118958
1,SumAmtL30D,0.117292
2,CountTrxTrf,0.080347
3,TotalTrxAmount10Mi,0.070671
4,CntUniqueCardNoByMCCL30D,0.064101
5,TotalTrxAmount15Mi,0.059420
6,TotalTrxAmountL5min,0.056407
7,TotalTrxAmountL1D,0.047092
8,TransactionAmount,0.045072
9,CustomerAge,0.044310
